In [1]:
%pip install "numpy<2" "opencv-python<5" "opencv-contrib-python<5" "mediapipe==0.10.14" pyautogui pandas

Note: you may need to restart the kernel to use updated packages.


In [3]:
import cv2
import pandas as pd
import mediapipe as mp
import pyautogui as pgui
pgui.FAILSAFE = False
import numpy as np
import math
import time

In [37]:
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 600)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 500)
screen_width, screen_height = pgui.size()

mp_drawing = mp.solutions.drawing_utils
mp_hands = mp.solutions.hands
hand = mp_hands.Hands(model_complexity=0,
    min_detection_confidence=0.93,
    min_tracking_confidence=0.93)

while True:
    success, frame = cap.read()
    height, width, _ = frame.shape
    mirror = cv2.flip(frame, 1)
    if success:
        RGB_frame = cv2.cvtColor(mirror, cv2.COLOR_BGR2RGB)
        result = hand.process(RGB_frame)
        if result.multi_hand_landmarks:
            for hand_landmarks in result.multi_hand_landmarks:
                mp_drawing.draw_landmarks(mirror, hand_landmarks, mp_hands.HAND_CONNECTIONS)
                index_fingertip = hand_landmarks.landmark[8]
                tip_x, tip_y = int(index_fingertip.x*width), int(index_fingertip.y*height)
                index_fingerpip = hand_landmarks.landmark[5]
                pip_x, pip_y = int(index_fingerpip.x*width), int(index_fingerpip.y*height)
                dx = tip_x - pip_x
                dy = tip_y - pip_y
                scale = 15
                xlinestart = int(tip_x - dx*scale)
                ylinestart = int(tip_y - dy*scale)
                xlineend = int(tip_x + dx*scale)
                ylineend = int(tip_y + dy*scale)
                cv2.line(mirror, (xlinestart, ylinestart), (xlineend, ylineend), (255, 0, 0), 2)
                if abs(index_fingertip.z - index_fingerpip.z) < 1e-06:
                    continue
                u = -(index_fingerpip.z)/(index_fingertip.z - index_fingerpip.z)
                intx = index_fingertip.x + u*(index_fingerpip.x - index_fingertip.x)
                inty = index_fingertip.y + u*(index_fingerpip.y - index_fingertip.y)
                mousex = int(np.interp(intx, [0, 1], [0, screen_width - 1]))
                mousey = int(np.interp(inty, [0, 1], [0, screen_height - 1]))
                pgui.moveTo(mousex, mousey, 0.1)
                thumbtip = hand_landmarks.landmark[4]
                thumb_x, thumb_y = int(thumbtip.x*width), int(thumbtip.y*height)
                pinky = hand_landmarks.landmark[12]
                p_x, p_y = int(pinky.x*width), int(pinky.y*height)
                cv2.line(mirror, (p_x, p_y), (thumb_x, thumb_y), (0, 255, 0), 2)
                clicklength = math.hypot(thumb_x - p_x, thumb_y - p_y)
                mouse_is_down = False
                if clicklength <= 40:
                    pgui.mouseDown(button = 'left')
                    mouse_is_down == True
                    # print("DOWN") 
                if clicklength > 40:
                    pgui.mouseUp(button = 'left')
                    mouse_is_down == False
                    # print("UP")
                right = hand_landmarks.landmark[5]
                r_x, r_y = int(right.x*width), int(right.y*height)
                cv2.line(mirror, (r_x, r_y), (thumb_x, thumb_y), (0, 255, 255), 2)
                Rclicklength = math.hypot(thumb_x - r_x, thumb_y - r_y)
                if Rclicklength <= 20:
                    pgui.rightClick()
                # print(Rclicklength)
        cv2.imshow("Live Feed", mirror)
        if cv2.waitKey(1) == ord('q'):
            cap.release()
            cv2.destroyAllWindows()
            break